# Modul 6 - Spasial Filter
Notebook ini berisi implementasi Praktikum, Percobaan, dan Tugas dari Modul 6.


In [ ]:
NAMA = 'Mohammad Al Hafis Hidayatulloh'
NIM = '254107023005'
print(NAMA, NIM)


## Persiapan Library & Mount Google Drive
Pastikan untuk menyesuaikan `drive_path` dengan folder Google Drive Anda.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import math
import torch
import torch.nn as nn
from PIL import Image as im
import os

# Silakan sesuaikan path dengan folder tempat Anda menyimpan gambar-gambar dari modul ini
drive_path = '/content/drive/MyDrive/pcvk_2026/'


## D. PRAKTIKUM FILTER
### 1. Fungsi Konvolusi Custom (Tanpa Library filter2D)


In [ ]:
def convolution2d(image, kernel, stride=1, padding=0):
    if padding > 0:
        image = np.pad(image, padding, mode='constant', constant_values=0)
    
    img_height, img_width = image.shape
    kernel_height, kernel_width = kernel.shape
    
    out_height = int((img_height - kernel_height) / stride) + 1
    out_width = int((img_width - kernel_width) / stride) + 1
    
    output = np.zeros((out_height, out_width), dtype=np.float32)
    
    for y in range(0, out_height):
        for x in range(0, out_width):
            y_img = y * stride
            x_img = x * stride
            roi = image[y_img:y_img+kernel_height, x_img:x_img+kernel_width]
            output[y, x] = np.sum(roi * kernel)
            
    return output


### 2. Load Citra (Grayscale) & Testing Praktikum Filter


In [ ]:
def show_side_by_side(images, titles, figsize=(15, 5)):
    plt.figure(figsize=figsize)
    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(1, len(images), i+1)
        if len(img.shape) == 2:
            plt.imshow(img, cmap="gray")
        else:
            plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
        plt.title(title)
        plt.axis("off")
    plt.show()

img_path = os.path.join(drive_path, 'mandrill.tiff')
img = cv.imread(img_path)

if img is not None:
    img_gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    
    kernel_sharpen = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    res_sharpen = convolution2d(img_gray, kernel_sharpen, 1, 1)
    res_sharpen = np.clip(res_sharpen, 0, 255).astype(np.uint8)

    kernel_emboss = np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]])
    res_emboss = convolution2d(img_gray, kernel_emboss, 1, 1)
    res_emboss = np.clip(res_emboss, 0, 255).astype(np.uint8)
    
    kernel_sobel = np.array([[1, 0, -1], [2, 0, -2], [1, 0, -1]])
    res_sobel = convolution2d(img_gray, kernel_sobel, 1, 1)
    res_sobel = np.clip(res_sobel, 0, 255).astype(np.uint8)
    
    kernel_canny = np.array([[-1, -1, -1], [-1, 8, -1], [-1, -1, -1]])
    res_canny = convolution2d(img_gray, kernel_canny, 1, 1)
    res_canny = np.clip(res_canny, 0, 255).astype(np.uint8)
    
    kernel_size = 21
    sigma = math.sqrt(kernel_size)
    gaussian_kernel_1d = cv.getGaussianKernel(kernel_size, sigma)
    gaussian_kernel_2d = gaussian_kernel_1d @ gaussian_kernel_1d.transpose()
    res_gaussian = convolution2d(img_gray, gaussian_kernel_2d, 1, kernel_size//2)
    res_gaussian = np.clip(res_gaussian, 0, 255).astype(np.uint8)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.ravel()
    titles_prak = ['Asli (Grayscale)', 'Sharpen', 'Emboss', 'Left Sobel', 'Canny Edge (Approx)', '21x21 Gaussian']
    images_prak = [img_gray, res_sharpen, res_emboss, res_sobel, res_canny, res_gaussian]
    
    for i in range(6):
        axes[i].imshow(images_prak[i], cmap='gray')
        axes[i].set_title(titles_prak[i])
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("Gambar mandrill.tiff tidak ditemukan")


## E. FILTER LIBRARY DAN FILTER MODERN
### Percobaan 1: Filter2D & OpenCV Filters


In [ ]:
img_lena_path = os.path.join(drive_path, 'lena.jpg')
img_lena = cv.imread(img_lena_path)

if img_lena is not None:
    blur = cv.GaussianBlur(img_lena, (7,7), 1)
    edges = cv.Canny(cv.cvtColor(img_lena, cv.COLOR_BGR2GRAY), 100, 200)
    sharpen_kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    sharpened = cv.filter2D(img_lena, -1, sharpen_kernel)
    
    show_side_by_side([img_lena, blur, sharpened, edges], 
                      ["Asli", "Gaussian Blur", "Sharpened", "Canny Edge Detection"])
else:
    print("Gambar lena.jpg tidak ditemukan!")


### Percobaan 2: Bilateral Filtering & Guided Filter


In [ ]:
if img_lena is not None:
    bilateral = cv.bilateralFilter(img_lena, 50, 100, 100)
    edge_preserve = cv.edgePreservingFilter(img_lena, flags=1, sigma_s=100, sigma_r=0.9)
    
    show_side_by_side([img_lena, bilateral, edge_preserve],
                      ["Asli", "Bilateral Filter", "Edge Preserving Filter"])


### Percobaan 3: Filtering pada CNN (Feature Map)


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 4, kernel_size=3, stride=1, padding=1)
        
    def forward(self, x):
        return self.conv1(x)

if img_lena is not None:
    model = SimpleCNN()
    img_gray_lena = cv.cvtColor(img_lena, cv.COLOR_BGR2GRAY)
    
    img_tensor = torch.tensor(img_gray_lena, dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0
    
    with torch.no_grad():
        features = model(img_tensor)
        
    feature_maps = [features[0, i].numpy() for i in range(features.shape[1])]
    
    show_side_by_side([img_gray_lena] + feature_maps, 
                      ["Asli (Gray)"] + [f"Feature {i+1}" for i in range(len(feature_maps))])


### Percobaan 4 & 5: Beauty, Vintage, dan Cartoon Filter


In [ ]:
if img_lena is not None:
    # --- Beauty Filter ---
    smooth = cv.bilateralFilter(img_lena, d=15, sigmaColor=75, sigmaSpace=75)
    gaussian = cv.GaussianBlur(smooth, (0,0), 3)
    sharpened_beauty = cv.addWeighted(smooth, 1.5, gaussian, -0.5, 0)
    alpha, beta = 1.2, 15
    beauty = cv.convertScaleAbs(sharpened_beauty, alpha=alpha, beta=beta)
    
    # --- Vintage Filter ---
    sepia_kernel = np.array([[0.272, 0.534, 0.131],
                             [0.349, 0.686, 0.168],
                             [0.393, 0.769, 0.189]])
    sepia = cv.transform(img_lena, sepia_kernel)
    sepia = np.clip(sepia, 0, 255).astype(np.uint8)
    
    rows, cols = img_lena.shape[:2]
    kernel_x = cv.getGaussianKernel(cols, cols*0.6)
    kernel_y = cv.getGaussianKernel(rows, rows*0.6)
    kernel_vignette = kernel_y * kernel_x.T
    mask = kernel_vignette / kernel_vignette.max()
    vignette = np.copy(sepia)
    for i in range(3):
        vignette[:,:,i] = vignette[:,:,i] * mask
        
    noise = np.random.normal(0, 15, vignette.shape).astype(np.int16)
    old_img = np.clip(vignette.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    
    # --- Cartoon Filter ---
    gray = cv.cvtColor(img_lena, cv.COLOR_BGR2GRAY)
    gray_blur = cv.medianBlur(gray, 7)
    edges_cartoon = cv.adaptiveThreshold(gray_blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, cv.THRESH_BINARY, 9, 9)
    color = cv.bilateralFilter(img_lena, d=9, sigmaColor=200, sigmaSpace=200)
    cartoon = cv.bitwise_and(color, color, mask=edges_cartoon)
    
    show_side_by_side([img_lena, beauty, old_img], ["Asli", "Beauty Filter", "Old/Vintage Filter"])
    show_side_by_side([img_lena, cartoon], ["Asli", "Cartoon Filter"])


### Percobaan 6 & 7: Filter Malam & Filter Pagi (Kabut)


In [ ]:
img_djawatan_path = os.path.join(drive_path, 'djawatan.jpg')
img_djawatan = cv.imread(img_djawatan_path)

if img_djawatan is not None:
    # --- Filter Malam ---
    night = cv.convertScaleAbs(img_djawatan, alpha=0.6, beta=-40)
    blue_tint = np.full_like(night, (50, 0, 100)) # BGR
    night = cv.addWeighted(night, 0.8, blue_tint, 0.2, 0)
    kernel_glow = np.ones((15,15), np.float32) / 225
    glow = cv.filter2D(night, -1, kernel_glow)
    night_glow = cv.addWeighted(night, 0.7, glow, 0.3, 0)
    
    show_side_by_side([img_djawatan, night, night_glow], 
                      ["Asli", "Night Filter", "Night + Glow (filter2D)"])
    
    # --- Filter Pagi dan Kabut ---
    soft = cv.convertScaleAbs(img_djawatan, alpha=0.9, beta=20)
    warm_tint = np.full_like(soft, (40, 70, 120))
    pagi = cv.addWeighted(soft, 0.8, warm_tint, 0.2, 0)
    
    kernel_blur_pagi = cv.getGaussianKernel(3, 3)
    kernel_blur_2d_pagi = kernel_blur_pagi @ kernel_blur_pagi.T
    kabut = cv.filter2D(pagi, -1, kernel_blur_2d_pagi)
    white_layer = np.full_like(pagi, 255)
    kabut = cv.addWeighted(kabut, 0.7, white_layer, 0.3, 0)
    
    show_side_by_side([img_djawatan, pagi, kabut], 
                      ["Asli", "Pagi", "Pagi + Kabut"])
else:
    print("Gambar djawatan.jpg tidak ditemukan")


## TUGAS PRAKTIKUM
### Tugas 1: Analisis: Perbandingan Mean Filter vs. Median Filter pada Noise Salt and Pepper


In [ ]:
def add_salt_and_pepper_noise(image, amount=0.05):
    row, col = image.shape
    out = np.copy(image)
    num_salt = np.ceil(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
    out[tuple(coords)] = 255
    num_pepper = np.ceil(amount * image.size * 0.5)
    coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
    out[tuple(coords)] = 0
    return out

img_task1 = cv.imread(os.path.join(drive_path, 'mandrill.tiff'), cv.IMREAD_GRAYSCALE)
if img_task1 is None:
    img_task1 = cv.imread(os.path.join(drive_path, 'lena.jpg'), cv.IMREAD_GRAYSCALE)

if img_task1 is not None:
    img_noisy = add_salt_and_pepper_noise(img_task1, amount=0.05)
    mean_filtered = cv.blur(img_noisy, (3, 3))
    median_filtered = cv.medianBlur(img_noisy, 3)
    
    show_side_by_side([img_task1, img_noisy, mean_filtered, median_filtered],
                      ["Asli", "Noisy", "Mean Filter", "Median Filter"])


**c. Analisis Visual Tugas 1:**
Berdasarkan hasil visual di atas, **Median Filter jauh lebih bersih dalam menghapus noise Salt and Pepper dibandingkan Mean Filter**. 
Hal ini terjadi karena:
1. **Mean Filter** menghitung nilai rata-rata dari piksel-piksel tetangga. Ketika ada noise dengan nilai ekstrem (hitam/0 atau putih/255), nilai tersebut akan ditarik ke rata-rata, sehingga noise tidak hilang melainkan menyebar dan membuat area di sekitarnya menjadi kabur/buram.
2. **Median Filter** mengurutkan nilai-nilai piksel di jendela kernel dan mengambil nilai tengahnya. Karena noise Salt and Pepper selalu bernilai ekstrem, nilai-nilai noise akan berada di ujung urutan dan nilai median yang terpilih akan selalu merupakan nilai piksel gambar yang wajar (bukan noise). Oleh karena itu, Median Filter bisa 'membuang' noise tanpa menyebarkannya.


### Tugas 2: Pemilihan Ukuran Kernel terbaik untuk Menajamkan Citra (Sharpening)


In [ ]:
img_task2 = cv.imread(os.path.join(drive_path, 'foto_bebas1.jpg'))
if img_task2 is None:
    img_task2 = cv.imread(os.path.join(drive_path, 'lena.jpg'))

if img_task2 is not None:
    img_blurry = cv.GaussianBlur(img_task2, (5, 5), 0)
    
    kernel_3x3 = np.array([[ 0, -1,  0],
                           [-1,  5, -1],
                           [ 0, -1,  0]])
    
    kernel_5x5 = np.array([[-1, -1, -1, -1, -1],
                           [-1,  2,  2,  2, -1],
                           [-1,  2,  8,  2, -1],
                           [-1,  2,  2,  2, -1],
                           [-1, -1, -1, -1, -1]]) / 8.0 
    
    sharpened_3x3 = cv.filter2D(img_blurry, -1, kernel_3x3)
    sharpened_5x5 = cv.filter2D(img_blurry, -1, kernel_5x5)
    
    show_side_by_side([img_blurry, sharpened_3x3, sharpened_5x5],
                      ["Citra Buram", "Sharpen 3x3", "Sharpen 5x5"])


**c. Evaluasi Tugas 2:**
- **Kejelasan Tepi:** Kernel 5x5 memiliki area jangkauan (*receptive field*) yang lebih luas sehingga efek kontras tepi menjadi lebih kuat. Namun, terkadang bisa menimbulkan efek *halo* (garis putih) yang mengelilingi objek karena ukurannya yang lebih besar. Kernel 3x3 menajamkan detail halus dengan lebih natural.
- **Kadar Noise/Noise Distortion:** Semakin besar ukuran kernel filter sharpening (contoh: 5x5), maka noise pada gambar akan semakin diperkuat secara eksponensial. Terlihat bahwa pada kernel 5x5, noise lebih terekspos dan warna terlihat *distorted*.
- **Kesimpulan:** Untuk penggunaan sehari-hari, **Kernel 3x3 lebih optimal**. Kernel ini dapat mengembalikan ketajaman citra buram dengan baik sambil menjaga tingkat noise dan efek *halo* agar tetap dalam batas wajar.


### Tugas 3: Kreasi Filter Kartun Sederhana


In [ ]:
def kartun(image):
    color = image
    for _ in range(2):
        color = cv.bilateralFilter(color, d=9, sigmaColor=250, sigmaSpace=250)
        
    gray = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    gray_blur = cv.medianBlur(gray, 7)
    edges = cv.adaptiveThreshold(gray_blur, 255, cv.ADAPTIVE_THRESH_MEAN_C, 
                                 cv.THRESH_BINARY, 9, 9)
    
    cartoon_img = cv.bitwise_and(color, color, mask=edges)
    return cartoon_img

img_task3 = cv.imread(os.path.join(drive_path, 'foto_benda.jpeg'))
if img_task3 is None:
    img_task3 = cv.imread(os.path.join(drive_path, 'lena.jpg'))

if img_task3 is not None:
    scale_percent = 50 
    width = int(img_task3.shape[1] * scale_percent / 100)
    height = int(img_task3.shape[0] * scale_percent / 100)
    dim = (width, height)
    img_task3_resized = cv.resize(img_task3, dim, interpolation = cv.INTER_AREA)

    img_kartun = kartun(img_task3_resized)
    
    show_side_by_side([img_task3_resized, img_kartun], ["Citra Asli", "Filter Kartun Sederhana"])
